# Kaggle: Combined Two-Dataset Training

This notebook is designed for Kaggle GPU notebooks. It clones the project into `/kaggle/working`, locates two attached Kaggle datasets, configures combined training, saves per-epoch checkpoints, evaluates the model, and creates a zip archive in `/kaggle/working`.

In [ ]:
REPO_URL = 'https://github.com/au510621104021/FND_2027.git'
BRANCH = 'main'
PROJECT_DIR = '/kaggle/working/FND_2027'

# Kaggle dataset search root. Attach your dataset(s) to the notebook first.
DATASET_SEARCH_ROOT = '/kaggle/input'

# Name hints for the two datasets to combine.
DATASET_1_HINTS = ['isot fake news dataset', 'isot']
DATASET_2_HINTS = ['fake news dataset']

# If auto-discovery selects the wrong folders, replace these with exact Kaggle paths.
DATASET_DIR_1 = None
DATASET_DIR_2 = None

OUTPUT_DIR = '/kaggle/working/outputs'

In [ ]:
import os
from pathlib import Path

os.chdir('/kaggle/working')
!rm -rf /kaggle/working/FND_2027
!git clone --branch main https://github.com/au510621104021/FND_2027.git /kaggle/working/FND_2027

assert Path(PROJECT_DIR).exists(), f'Project directory not found: {PROJECT_DIR}'
os.chdir(PROJECT_DIR)
print('Project directory:', os.getcwd())
!ls

In [ ]:
import os
from pathlib import Path

os.chdir(PROJECT_DIR)
assert Path('requirements.txt').exists(), 'requirements.txt not found. Clone step likely failed.'
!pip install -q -r requirements.txt

In [ ]:
from pathlib import Path

CSV_NAMES = {'train.csv', 'test.csv', 'dataset.csv', 'data.csv', 'Fake.csv', 'True.csv'}

def folder_has_dataset_csvs(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    names = {p.name for p in path.glob('*.csv')}
    return bool(names.intersection(CSV_NAMES))

def collect_dataset_candidates(search_root: str):
    root = Path(search_root)
    candidates = []
    seen = set()
    if not root.exists():
        return candidates
    for csv_file in root.rglob('*.csv'):
        parent = csv_file.parent.resolve()
        if str(parent) in seen:
            continue
        if folder_has_dataset_csvs(parent):
            seen.add(str(parent))
            candidates.append(parent)
    return sorted(candidates, key=lambda p: str(p).lower())

def pick_candidate(candidates, hints):
    scored = []
    for path in candidates:
        text = str(path).lower()
        score = 0
        for hint in hints:
            if hint in path.name.lower():
                score += 5
            if hint in text:
                score += 2
        if score > 0:
            scored.append((score, len(path.parts), str(path)))
    if not scored:
        return None
    scored.sort(key=lambda x: (-x[0], x[1], x[2]))
    return scored[0][2]

candidates = collect_dataset_candidates(DATASET_SEARCH_ROOT)
print('Detected Kaggle dataset candidate folders:')
for candidate in candidates:
    print(' -', candidate)

if DATASET_DIR_1 is None:
    DATASET_DIR_1 = pick_candidate(candidates, DATASET_1_HINTS)
if DATASET_DIR_2 is None:
    DATASET_DIR_2 = pick_candidate(candidates, DATASET_2_HINTS)

print('\nSelected DATASET_DIR_1 =', DATASET_DIR_1)
print('Selected DATASET_DIR_2 =', DATASET_DIR_2)

assert DATASET_DIR_1 is not None, 'Could not find dataset 1 automatically. Use one of the printed candidate folders.'
assert DATASET_DIR_2 is not None, 'Could not find dataset 2 automatically. Use one of the printed candidate folders.'
assert Path(DATASET_DIR_1).exists(), f'Folder not found: {DATASET_DIR_1}'
assert Path(DATASET_DIR_2).exists(), f'Folder not found: {DATASET_DIR_2}'

print('\nDataset 1 CSVs:', [p.name for p in Path(DATASET_DIR_1).glob('*.csv')])
print('Dataset 2 CSVs:', [p.name for p in Path(DATASET_DIR_2).glob('*.csv')])

In [ ]:
import os
import yaml
from pathlib import Path

os.chdir(PROJECT_DIR)
config_path = Path('config/config.yaml')
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['dataset_name'] = 'generic'
config['data']['data_dir'] = DATASET_DIR_1
config['data']['dataset_names'] = ['generic', 'generic']
config['data']['data_dirs'] = [DATASET_DIR_1, DATASET_DIR_2]
config['logging']['checkpoint_dir'] = '/kaggle/working/checkpoints'
config['logging']['log_dir'] = '/kaggle/working/logs'
config['inference']['model_checkpoint'] = '/kaggle/working/checkpoints/best_model_combined.pt'
config['publication']['results_dir'] = '/kaggle/working/results'

with config_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config for Kaggle combined training:')
print(config['data'])

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/train.py --config config/config.yaml

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir(PROJECT_DIR)
best_model = Path('/kaggle/working/checkpoints/best_model.pt')
combined_model = Path('/kaggle/working/checkpoints/best_model_combined.pt')
assert best_model.exists(), 'Training did not produce /kaggle/working/checkpoints/best_model.pt'
shutil.copy2(best_model, combined_model)
print(f'Saved combined checkpoint to: {combined_model}')

In [ ]:
import os
os.chdir(PROJECT_DIR)
!python scripts/evaluate.py --config config/config.yaml --checkpoint /kaggle/working/checkpoints/best_model_combined.pt

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir('/kaggle/working')
os.makedirs(OUTPUT_DIR, exist_ok=True)

export_dir = Path(OUTPUT_DIR) / 'combined_model_export'
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

for src_name in ['checkpoints', 'logs', 'results']:
    src_path = Path('/kaggle/working') / src_name
    if src_path.exists():
        dst_path = export_dir / src_name
        shutil.copytree(src_path, dst_path)

archive_path = shutil.make_archive('/kaggle/working/combined_model_artifacts', 'zip', export_dir)
print('Created zip:', archive_path)
print('You can download it from the Kaggle notebook output pane after the run finishes.')